In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from tqdm import tqdm
import re
import os

# 关键：禁用tokenizers并行，避免多进程fork死锁问题
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
def load_and_preprocess_data():
    # 1. 加载竞赛数据（请确保train.csv和test.csv在当前路径下）
    train_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")
    test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")
    
    # 2. 安全解析文本（处理字符串列表格式，如"['prompt1', 'prompt2']"）
    def safe_eval(text):
        try:
            # 替换null为空字符串，避免解析报错
            text = re.sub(r'null', "''", text)
            parsed = eval(text)  # 解析字符串列表
            # 取第一个元素（竞赛数据中通常第一个为有效内容），确保返回字符串
            if isinstance(parsed, list) and len(parsed) > 0:
                return str(parsed[0])
            return ""
        except Exception as e:
            # 解析失败时返回空字符串，避免中断流程
            return ""
    
    # 3. 批量处理所有文本列（prompt、response_a、response_b）
    for df in [train_df, test_df]:
        # 解析文本
        df["prompt"] = df["prompt"].apply(safe_eval)
        df["response_a"] = df["response_a"].apply(safe_eval)
        df["response_b"] = df["response_b"].apply(safe_eval)
        
        # 4. 清理非法UTF-8字符（核心解决编码错误）
        def clean_utf8(text):
            # 用"�"替换无法编码的字符，再解码为合法UTF-8
            return text.encode('utf-8', errors='replace').decode('utf-8')
        
        df["prompt"] = df["prompt"].fillna("").astype(str).apply(clean_utf8)
        df["response_a"] = df["response_a"].fillna("").astype(str).apply(clean_utf8)
        df["response_b"] = df["response_b"].fillna("").astype(str).apply(clean_utf8)
    
    # 5. 生成训练标签（将3个二进制标签转为单标签：0=A赢，1=B赢，2=平局）
    if "winner_model_a" in train_df.columns:
        train_df["label"] = train_df[["winner_model_a", "winner_model_b", "winner_tie"]].idxmax(axis=1).map(
            {"winner_model_a": 0, "winner_model_b": 1, "winner_tie": 2}
        )
    
    return train_df, test_df

# 执行数据预处理（这一步会生成清洁后的训练/测试数据）
train_df, test_df = load_and_preprocess_data()
print("数据预处理完成！训练集行数：", len(train_df), "测试集行数：", len(test_df))

In [ ]:
class LLMPreferenceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df  # 输入数据框
        self.tokenizer = tokenizer  # 分词器
        self.max_len = max_len  # 文本最大长度（适配模型）
        
        # 1. 构建模型输入文本（拼接prompt+response_a+response_b，模拟"问题+两个选项"）
        self.texts = []
        for _, row in df.iterrows():
            text = f"Prompt: {row['prompt']}\n\nResponse A: {row['response_a']}\n\nResponse B: {row['response_b']}"
            text = re.sub(r'\s+', ' ', text).strip()  # 合并多余空格，避免格式混乱
            self.texts.append(text)
        
        # 2. 提取标签（测试集无标签，设为None）
        self.labels = df["label"].tolist() if "label" in df.columns else None

    def __len__(self):
        # 返回数据集总条数
        return len(self.df)

    def __getitem__(self, idx):
        # 按索引获取单条数据，转为模型输入格式
        text = self.texts[idx]
        
        # 安全校验：确保文本是有效字符串
        if not isinstance(text, str) or len(text) == 0:
            text = " "  # 空文本用空格代替
        
        try:
            # 3. Token化处理（模型输入的核心步骤）
            inputs = self.tokenizer(
                text,
                max_length=self.max_len,  # 截断/补齐到固定长度
                padding="max_length",     # 自动补齐
                truncation=True,          # 超长截断
                return_tensors="pt"       # 返回PyTorch张量
            )
            # 移除多余的batch维度（Tokenizer默认返回(batch, seq_len)，需转为(seq_len)）
            inputs = {k: v.squeeze(0) for k, v in inputs.items()}
            
            # 4. 添加标签（训练集用，测试集无）
            if self.labels is not None:
                inputs["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            
            return inputs
        
        except Exception as e:
            # 异常处理：打印问题文本，避免流程中断
            print(f"警告：处理索引{idx}时出错：{str(e)}")
            print(f"问题文本（前100字符）：{repr(text[:100])}")
            
            # 极端情况：用清洁后的文本重试
            cleaned_text = re.sub(r'[^\x00-\x7F]+', '', text)  # 移除所有非ASCII字符
            cleaned_text = cleaned_text[:1000]  # 限制最大长度
            inputs = self.tokenizer(
                cleaned_text,
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            inputs = {k: v.squeeze(0) for k, v in inputs.items()}
            
            if self.labels is not None:
                inputs["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            
            return inputs

In [ ]:
# 1. 模型配置（可根据资源调整：小模型=deberta-v3-small，中模型=deberta-v3-base）
model_name = "microsoft/deberta-v3-small"  # 平衡性能与速度的选择
max_len = 512  # 文本最大长度（显存不足可减为256）
batch_size = 4  # 批次大小（显存不足可减为2）

# 2. 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 加载分类模型（num_labels=3：A赢、B赢、平局）
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 3. 划分训练集与验证集（按标签分层，避免数据分布不均）
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,  # 20%数据作为验证集
    random_state=42,  # 固定随机种子，结果可复现
    stratify=train_df["label"]  # 按标签分层，确保验证集标签分布与训练集一致
)

# 4. 构建Dataset实例
train_dataset = LLMPreferenceDataset(train_data, tokenizer, max_len)
val_dataset = LLMPreferenceDataset(val_data, tokenizer, max_len)

# 5. 构建DataLoader（数据迭代器，核心优化：禁用多进程避免编码冲突）
# num_workers=0：单进程加载，虽然慢但稳定（多进程易出现编码/死锁问题）
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # 训练集打乱，提升泛化能力
    num_workers=0
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,  # 验证集不打乱，结果可复现
    num_workers=0
)

# 6. 设备配置（自动使用GPU/CPU）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # 将模型移到GPU/CPU
print(f"模型已加载到{device}，训练集批次数量：{len(train_loader)}，验证集批次数量：{len(val_loader)}")

In [ ]:
def train_model(model, train_loader, val_loader, device, epochs=3):
    # 1. 优化器配置（AdamW+权重衰减，防止过拟合）
    optimizer = AdamW(
        model.parameters(),
        lr=2e-5,  # 学习率（小模型推荐1e-5~3e-5）
        weight_decay=0.01  # 权重衰减，抑制过拟合
    )
    
    # 2. 学习率调度器（预热+线性衰减，稳定训练）
    total_steps = len(train_loader) * epochs  # 总训练步数
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),  # 10%步数用于预热（从低学习率开始）
        num_training_steps=total_steps  # 总步数
    )
    
    # 3. 训练循环（保存验证损失最小的模型）
    best_val_loss = float("inf")  # 初始最佳损失设为无穷大
    for epoch in range(epochs):
        # ---------------------- 训练阶段 ----------------------
        model.train()  # 模型设为训练模式（启用dropout等）
        train_loss = 0.0
        # 迭代训练集
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} | 训练"):
            # 数据移到设备（GPU/CPU）
            batch = {k: v.to(device) for k, v in batch.items()}
            
            # 梯度清零（避免累积）
            optimizer.zero_grad()
            # 前向传播（模型预测）
            outputs = model(** batch)
            # 计算损失（分类交叉熵，适配3分类）
            loss = outputs.loss
            # 反向传播（计算梯度）
            loss.backward()
            # 梯度裁剪（防止梯度爆炸，核心稳定训练）
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            # 更新参数（优化器.step()）
            optimizer.step()
            # 更新学习率（调度器.step()）
            scheduler.step()
            
            # 累积训练损失
            train_loss += loss.item() * batch["input_ids"].size(0)
        
        # 计算平均训练损失
        avg_train_loss = train_loss / len(train_loader.dataset)
        
        # ---------------------- 验证阶段 ----------------------
        model.eval()  # 模型设为评估模式（禁用dropout等）
        val_loss = 0.0
        val_preds = []  # 验证集预测概率
        val_true = []   # 验证集真实标签
        with torch.no_grad():  # 禁用梯度计算，节省显存和时间
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} | 验证"):
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(** batch)
                loss = outputs.loss
                val_loss += loss.item() * batch["input_ids"].size(0)
                
                # 计算预测概率（softmax转为0-1概率）
                logits = outputs.logits
                preds = torch.softmax(logits, dim=1).cpu().numpy()
                val_preds.extend(preds)
                # 收集真实标签
                val_true.extend(batch["labels"].cpu().numpy())
        
        # 计算验证集指标
        avg_val_loss = val_loss / len(val_loader.dataset)
        val_log_loss = log_loss(val_true, val_preds)  # 竞赛评估指标：Log Loss
        
        # ---------------------- 日志与保存 ----------------------
        print(f"\nEpoch {epoch+1} 结果：")
        print(f"训练损失：{avg_train_loss:.4f} | 验证损失：{avg_val_loss:.4f} | 验证Log Loss：{val_log_loss:.4f}")
        
        # 保存最佳模型（验证损失最小的模型，避免过拟合）
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_model.pt")  # 保存模型权重
            print(f"✅ 保存最佳模型（验证损失：{best_val_loss:.4f}）")
    
    return model

# 执行训练（默认3轮，显存充足可增至5轮）
trained_model = train_model(model, train_loader, val_loader, device, epochs=3)

In [ ]:
def inference_and_submit(model, test_df, tokenizer, max_len=512, batch_size=8):
    # 1. 加载最佳模型权重（训练阶段保存的"best_model.pt"）
    model.load_state_dict(torch.load("best_model.pt"))
    model.to(device)
    model.eval()  # 设为评估模式
    
    # 2. 构建测试集Dataset和DataLoader
    test_dataset = LLMPreferenceDataset(test_df, tokenizer, max_len)
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    # 3. 测试集推理（禁用梯度计算，加速）
    test_preds = []  # 存储所有预测概率
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="测试集推理"):
            # 数据移到设备（测试集无标签，过滤掉labels键）
            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            outputs = model(** batch)
            # 计算预测概率（softmax后转为numpy数组）
            logits = outputs.logits
            preds = torch.softmax(logits, dim=1).cpu().numpy()
            test_preds.extend(preds)
    
    # 4. 生成提交文件（竞赛要求格式：id, winner_model_a, winner_model_b, winner_tie）
    sub_df = test_df[["id"]].copy()  # 提取测试集ID
    # 填入预测概率（3列分别对应A赢、B赢、平局）
    sub_df[["winner_model_a", "winner_model_b", "winner_tie"]] = pd.DataFrame(test_preds)
    # 保存为CSV（index=False：不保留行索引，符合竞赛要求）
    sub_df.to_csv("submission.csv", index=False)
    
    print(f"提交文件已生成！共{len(sub_df)}行，前5行预览：")
    print(sub_df.head())

# 执行推理与提交文件生成
inference_and_submit(trained_model, test_df, tokenizer, max_len=512, batch_size=8)